In [1]:
import sys
import os
import pandas as pd
import random
import copy
import pickle
import math
import numpy as np
import time
import torch
from sklearn.datasets import make_blobs
from sklearn.preprocessing import MinMaxScaler
from datasets.metadata import uci_datasets_info
from utils.process_data import preprocess_dataframe
from utils.clustering import *
from Walker import ActiveDPGMMWalker
from utils.measures import *

torch.manual_seed(20260421)
rng = np.random.default_rng(seed=20260421)

/home/bdezoysa/.conda/envs/py310/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [2]:
num_bins = 32

In [8]:
# pocket_sizes_dict = {
#     16000: [8, 16, 32, 64],
#     32000: [16, 32, 64, 128],
#     64000: [32, 64, 128, 256],
#     128000: [64, 128, 256, 512],
#     256000: [128, 256, 512, 1024],
#     512000: [256, 512, 1024, 2048],
#     1024000: [512, 1024, 2048, 4096]
# }

# pocket_sizes_dict = {
#     16000: [512, 1024, 2048, 4096],
#     32000: [512, 1024, 2048, 4096],
#     64000: [512, 1024, 2048, 4096],
#     128000: [512, 1024, 2048, 4096],
#     256000: [512, 1024, 2048, 4096],
#     512000: [512, 1024, 2048, 4096],
#     1024000: [512, 1024, 2048, 4096],
# }

pocket_sizes_dict = {
    16000: [512],
    32000: [512],
    64000: [512],
    128000: [512],
    256000: [512],
    512000: [512],
    1024000: [512],
}


for N in [16000, 32000, 64000, 128000, 256000, 512000, 1024000]:
    X, _ = make_blobs(n_samples=N, centers=2000, n_features=2, cluster_std=1.5, random_state=20260421)
    scaler = MinMaxScaler(feature_range=(0, 1))
    X_scaled = scaler.fit_transform(X)
    tar_x = rng.random((N, 1))
    Z = np.concatenate([tar_x, X_scaled], axis=1)
    df = pd.DataFrame(Z, columns=[f'x{i}' for i in range(3)])
    condition = ((df['x1'] > 0.9) & (df['x2'] < 0.3)) | ((df['x1'] < 0.2) & (df['x2'] > 0.9)) | ((0.4 < df['x1']) & (df['x1'] < 0.6) & (0.2 < df['x2']) & (df['x2'] < 0.4))
    filtered_df = df[condition]
    print(filtered_df.shape)
    filtered_indices = filtered_df.index
    df.loc[filtered_indices, 'x0'] = np.random.normal(loc=1.5, scale=0.5, size=len(filtered_indices))

    tensor_data = torch.tensor(df.values, dtype=torch.float32, device='cuda')

    tensor_data = (tensor_data - tensor_data.min(dim=0).values) / tensor_data.max(dim=0).values
    col_widths = (tensor_data.max(dim=0).values + 1e-6 - tensor_data.min(dim=0).values) / num_bins

    pos = tensor_data[:, 1:].contiguous()
    tar = tensor_data[:, 0].unsqueeze(1).contiguous()

    hist_idx = (tar // col_widths[0]).long().squeeze(-1)
    hists = torch.zeros((N, num_bins), device='cuda')
    hists[torch.arange(N), hist_idx] = 1.0

    for pocket_size in pocket_sizes_dict[N]:
        # if os.path.isfile(f'./perceptive_graphs/synthetic_N{N}_p{pocket_size}.pkl'):
        #     print(f'Skipping N={N} p={pocket_size}...')
        #     continue

        cl, c, duration = KMeans(pos, K=math.ceil(pos.shape[0] / pocket_size), Niter=10000, tol=1e-6)

        pockets = []
        for ci in cl.unique():
            pockets.append(torch.nonzero(cl == ci).squeeze(-1))

        cl_index = cl.unsqueeze(1).expand(-1, hists.shape[1])
        cl_hists = torch.zeros(pos.shape[0], hists.shape[1], device='cuda')
        cl_hists.scatter_add_(0, cl_index, hists)

        if len(pockets) < pos.shape[0]:
            cl_hists = cl_hists[cl.unique()]
            c = c[cl.unique()]

        with open(f'./perceptive_graphs/synthetic_N{N}_p{pocket_size}.pkl', 'wb') as f:
            pickle.dump({'pockets': pockets, 'midpoints': c, 'pocket_hists': cl_hists, 'time': duration}, f)

(1245, 3)
Converged at iteration 1523
K-means for the Manhattan metric with 16,000 points in dimension 2, K = 32:
Timing for 10000 iterations: 0.40721s = 10000 x 0.00004s

(2749, 3)
K-means for the Manhattan metric with 32,000 points in dimension 2, K = 63:
Timing for 10000 iterations: 2.73434s = 10000 x 0.00027s

(5608, 3)
K-means for the Manhattan metric with 64,000 points in dimension 2, K = 125:
Timing for 10000 iterations: 2.94059s = 10000 x 0.00029s

(10714, 3)
K-means for the Manhattan metric with 128,000 points in dimension 2, K = 250:
Timing for 10000 iterations: 3.56605s = 10000 x 0.00036s

(24165, 3)
K-means for the Manhattan metric with 256,000 points in dimension 2, K = 500:
Timing for 10000 iterations: 5.14783s = 10000 x 0.00051s

(47990, 3)
K-means for the Manhattan metric with 512,000 points in dimension 2, K = 1,000:
Timing for 10000 iterations: 11.47666s = 10000 x 0.00115s

(95606, 3)
K-means for the Manhattan metric with 1,024,000 points in dimension 2, K = 2,000:
Ti

In [23]:
for C in [16, 32, 64, 128, 256, 512, 1024]:
    X, _ = make_blobs(n_samples=16000, centers=2000, n_features=C, cluster_std=1.5, random_state=20260421)
    scaler = MinMaxScaler(feature_range=(0, 1))
    X_scaled = scaler.fit_transform(X)
    tar_x = rng.random((16000, 1))
    Z = np.concatenate([tar_x, X_scaled], axis=1)

    df = pd.DataFrame(Z, columns=[f'x{i}' for i in range(C+1)])
    condition = ((df['x1'] > 0.9) & (df['x2'] < 0.3)) | ((df['x1'] < 0.2) & (df['x2'] > 0.9)) | ((0.4 < df['x1']) & (df['x1'] < 0.6) & (0.2 < df['x2']) & (df['x2'] < 0.4))
    filtered_df = df[condition]
    print(filtered_df.shape)
    filtered_indices = filtered_df.index
    df.loc[filtered_indices, 'x0'] = np.random.normal(loc=1.5, scale=0.5, size=len(filtered_indices))

    tensor_data = torch.tensor(df.values, dtype=torch.float32, device='cuda')
    
    tensor_data = (tensor_data - tensor_data.min(dim=0).values) / tensor_data.max(dim=0).values
    col_widths = (tensor_data.max(dim=0).values + 1e-6 - tensor_data.min(dim=0).values) / num_bins

    pos = tensor_data[:, 1:].contiguous()
    tar = tensor_data[:, 0].unsqueeze(1).contiguous()

    hist_idx = (tar // col_widths[0]).long().squeeze(-1)
    hists = torch.zeros((16000, num_bins), device='cuda')
    hists[torch.arange(16000), hist_idx] = 1.0

    for pocket_size in [8, 16, 32, 64]:
        cl, c, duration = KMeans(pos, K=pos.shape[0] // pocket_size, Niter=10000)

        pockets = []
        for ci in cl.unique():
            pockets.append(torch.nonzero(cl == ci).squeeze(-1))

        cl_index = cl.unsqueeze(1).expand(-1, hists.shape[1])
        cl_hists = torch.zeros(pos.shape[0], hists.shape[1], device='cuda')
        cl_hists.scatter_add_(0, cl_index, hists)

        if len(pockets) < pos.shape[0]:
            cl_hists = cl_hists[cl.unique()]
            c = c[cl.unique()]

        with open(f'./perceptive_graphs/synthetic_C{C}_p{pocket_size}.pkl', 'wb') as f:
            pickle.dump({'pockets': pockets, 'midpoints': c, 'pocket_hists': cl_hists, 'time': duration}, f)

(1344, 17)
K-means for the Manhattan metric with 16,000 points in dimension 16, K = 2,000:
Timing for 10000 iterations: 3.91041s = 10000 x 0.00039s

K-means for the Manhattan metric with 16,000 points in dimension 16, K = 1,000:
Timing for 10000 iterations: 3.25142s = 10000 x 0.00033s

K-means for the Manhattan metric with 16,000 points in dimension 16, K = 500:
Timing for 10000 iterations: 2.97903s = 10000 x 0.00030s

K-means for the Manhattan metric with 16,000 points in dimension 16, K = 250:
Timing for 10000 iterations: 2.77814s = 10000 x 0.00028s

(1353, 33)
K-means for the Manhattan metric with 16,000 points in dimension 32, K = 2,000:
Timing for 10000 iterations: 5.22594s = 10000 x 0.00052s

K-means for the Manhattan metric with 16,000 points in dimension 32, K = 1,000:
Timing for 10000 iterations: 3.98729s = 10000 x 0.00040s

K-means for the Manhattan metric with 16,000 points in dimension 32, K = 500:
Timing for 10000 iterations: 3.32658s = 10000 x 0.00033s

K-means for the Ma